# Exploring Facebook Ads Dataset

This notebook shows my "explore" process before I cleaned the data. It has 1,143 Facebook ads from 3 campaigns. Each row shows who the ad targeted (age, gender, interest) and how it performed (impressions, clicks, money spent, and conversions). There were no missing values or duplicate rows, and every ad_id is unique.

My thought process and observations are written alongside the relevant code.

In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/KAG_conversion_data.csv")
df.info()
df.head()

<class 'pandas.DataFrame'>
RangeIndex: 1143 entries, 0 to 1142
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   ad_id                1143 non-null   int64  
 1   xyz_campaign_id      1143 non-null   int64  
 2   fb_campaign_id       1143 non-null   int64  
 3   age                  1143 non-null   str    
 4   gender               1143 non-null   str    
 5   interest             1143 non-null   int64  
 6   Impressions          1143 non-null   int64  
 7   Clicks               1143 non-null   int64  
 8   Spent                1143 non-null   float64
 9   Total_Conversion     1143 non-null   int64  
 10  Approved_Conversion  1143 non-null   int64  
dtypes: float64(1), int64(8), str(2)
memory usage: 98.4 KB


,ad_id,xyz_campaign_id,fb_campaign_id,age,gender,interest,Impressions,Clicks,Spent,Total_Conversion,Approved_Conversion
0,708746,916,103916,30-34,M,15,7350,1,1.43,2,1
1,708749,916,103917,30-34,M,16,17861,2,1.82,2,0
2,708771,916,103920,30-34,M,20,693,0,0.00,1,0
3,708815,916,103928,30-34,M,28,4259,1,1.25,1,0
4,708818,916,103928,30-34,M,28,4133,1,1.29,1,1


In [2]:
df.describe()

,ad_id,xyz_campaign_id,fb_campaign_id,interest,Impressions,Clicks,Spent,Total_Conversion,Approved_Conversion
count,1.143000e+03,1143.000000,1143.000000,1143.000000,1.143000e+03,1143.000000,1143.000000,1143.000000,1143.000000
mean,9.872611e+05,1067.382327,133783.989501,32.766404,1.867321e+05,33.390201,51.360656,2.855643,0.944007
std,1.939928e+05,121.629393,20500.308622,26.952131,3.127622e+05,56.892438,86.908418,4.483593,1.737708
min,7.087460e+05,916.000000,103916.000000,2.000000,8.700000e+01,0.000000,0.000000,0.000000,0.000000
25%,7.776325e+05,936.000000,115716.000000,16.000000,6.503500e+03,1.000000,1.480000,1.000000,0.000000
50%,1.121185e+06,1178.000000,144549.000000,25.000000,5.150900e+04,8.000000,12.370000,1.000000,1.000000
75%,1.121804e+06,1178.000000,144657.500000,31.000000,2.217690e+05,37.500000,60.025000,3.000000,1.000000
max,1.314415e+06,1178.000000,179982.000000,114.000000,3.052003e+06,421.000000,639.949998,60.000000,21.000000


**Skewed data:** A few ads have many more impressions than the others, which creates misleading averages. In the Impressions column, the mean is much higher than the median (50%). To account for this, I compared groups using rates instead of averages.

In [3]:
df.isnull().sum()

ad_id                  0
xyz_campaign_id        0
fb_campaign_id         0
age                    0
gender                 0
interest               0
Impressions            0
Clicks                 0
Spent                  0
Total_Conversion       0
Approved_Conversion    0
dtype: int64

In [4]:
df.duplicated().sum()

np.int64(0)

In [5]:
len(df)

1143

In [6]:
# Is there one row per ad_id?
df["ad_id"].nunique()

1143

In [7]:
df["xyz_campaign_id"].value_counts()

xyz_campaign_id
1178    625
936     464
916      54
Name: count, dtype: int64

In [8]:
df["age"].value_counts()

age
30-34    426
45-49    259
35-39    248
40-44    210
Name: count, dtype: int64

In [9]:
# How many ads have conversions (inquiries) but don't have clicks?
no_clicks = df["Clicks"] == 0
has_conversions = df["Total_Conversion"] > 0
(no_clicks & has_conversions).sum()

np.int64(204)

**Conversions without clicks:** 204 ads have conversions (inquiries) but zero clicks. At first I thought this could be an error and considered deleting these rows. I kept them because it's possible for users to see an ad and not click it, but inquire later.

In [10]:
# How many ads have approved conversions (purchases) but don't have clicks?
has_approved = df["Approved_Conversion"] > 0
(no_clicks & has_approved).sum()

np.int64(71)

71 ads also had approved conversions (purchases) without any clicks.

In [11]:
# Do any ads without clicks have spend?
zero_click_ads = df[df["Clicks"] == 0]
zero_click_ads["Spent"].max()

np.float64(0.0)

**Pay per click:** When an ad has zero clicks, it also has zero spend. This makes me think the ads were paid for per click (PPC).

In [12]:
# Do all of the ads within the same ad set target the same audience?
audience_cols = ["xyz_campaign_id", "age", "gender", "interest"]
audience_counts = df.groupby("fb_campaign_id")[audience_cols].nunique()
audience_counts.head()

,xyz_campaign_id,age,gender,interest
fb_campaign_id,,,,
103916,1,1,1,1
103917,1,1,1,1
103920,1,1,1,1
103928,1,1,1,1
103929,1,1,1,1


In [13]:
# If each ad set targets only one audience, then the max should be 1
audience_counts.max()

xyz_campaign_id    1
age                1
gender             1
interest           1
dtype: int64

**Ad sets:** Each fb_campaign_id always targets the same age, gender, and interest. In Meta's ads system, targeting is set at the ad set level (campaign → ad set → ad), so fb_campaign_id works more like an ad set than a campaign. I split the data into three tables (campaigns, ad sets, and ads) because of this.

In [14]:
# Total impressions, clicks, spend, and approved conversions per campaign
totals_cols = ["Impressions", "Clicks", "Spent", "Approved_Conversion"]
summary = df.groupby("xyz_campaign_id")[totals_cols].sum()
summary

,Impressions,Clicks,Spent,Approved_Conversion
xyz_campaign_id,,,,
916,482925,113,149.710001,24
936,8128187,1984,2893.369999,183
1178,204823716,36068,55662.149959,872


In [15]:
# Calculate rates from the campaign totals instead of averaging each ad's rate
summary["ctr_pct"] = summary["Clicks"] / summary["Impressions"] * 100
summary["cpc"] = summary["Spent"] / summary["Clicks"]
summary["cost_per_approved"] = summary["Spent"] / summary["Approved_Conversion"]
summary.round(3)

,Impressions,Clicks,Spent,Approved_Conversion,ctr_pct,cpc,cost_per_approved
xyz_campaign_id,,,,,,,
916,482925,113,149.71,24,0.023,1.325,6.238
936,8128187,1984,2893.37,183,0.024,1.458,15.811
1178,204823716,36068,55662.15,872,0.018,1.543,63.833


**Campaign 1178** used 94.8% of the total ad spend, but it had the worst cost per approved conversion ($63.83) and the lowest click-through rate.

**Campaign 916** had the cheapest cost per approved conversion ($6.24), but it only had 54 ads. This result may not hold up with more data because of the small number of ads.